In [1]:
import os
from dotenv import load_dotenv

load_dotenv()
os.environ['HF_Token']=os.getenv("HF_Token")

In [2]:
from langchain_chroma import Chroma
from langchain_community.document_loaders import TextLoader
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_text_splitters import CharacterTextSplitter

f:\Python\.conda\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [3]:
loader = TextLoader("Speech.txt")
data = loader.load()
data

[Document(metadata={'source': 'Speech.txt'}, page_content='The word ‘Hindu’ has been derived from the call of the cow called ‘hinkaar’. The cow’s hinkaar has given the name Hindu as per the Rigveda. The word ‘Hindu’ has been used as ‘hi’ and ‘ndu’, meaning cow-worshippers. In the same way (as is, in the stated acronym style), this has also been mentioned in the Atharvaveda.\n\n‘Sa’ and ‘ha’ are the same and can be used interchangeably. From this point of view, the word ‘Sindhusthana’ is used for ‘Hindusthana’ and this has been called the country of Aryas (noble people) as per the Bhavishya Purana.\n\nTaking into view, the region from Himalaya (Mansarovar) to Indu Sarovar (Kanyakumari), ‘hi’ and ‘ndu’ combine as an acronym to create the word ‘Hindu’, and define the geographical region is called Hindusthana or Hindustan\n\nThe one who overcomes deficiencies, poverty, meanness, pettiness, is a Hindu. This is the yogic meaning of the word.\n\nThe one who destroys ‘hinata’ or inferiority, i

In [4]:
text_splitter = CharacterTextSplitter(chunk_size=320, chunk_overlap = 40)
splits = text_splitter.split_documents(data)


In [5]:
embeddings = HuggingFaceEmbeddings()

vectordb = Chroma.from_documents(splits, embeddings)

Loading weights: 100%|██████████| 199/199 [00:00<00:00, 2380.01it/s]
MPNetModel LOAD REPORT from: sentence-transformers/all-mpnet-base-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [12]:
#query the db
query="Where is Indu Sarovar?"
docs = vectordb.similarity_search("Atharvaveda")
docs

[Document(id='59b04784-06be-4107-9917-31f8e90d17bc', metadata={'source': 'Speech.txt'}, page_content='In Sharngdhar Padhatti, the word ‘Hindvah’ has been used for people who follow the vedic path.\n\nIn the Brihaspati Agama the kshetra or region of Hindusthana has also been mentioned.\n\nIn the Ashvamedhika Parva of Mahabharata, the region of ‘Aryavarta’ has been called Hindusthanaa or Sindhusthanaa.'),
 Document(id='8055bba7-0aa7-4dda-bbf9-eae943d29d94', metadata={'source': 'Speech.txt'}, page_content='The word ‘Hindu’ has been derived from the call of the cow called ‘hinkaar’. The cow’s hinkaar has given the name Hindu as per the Rigveda. The word ‘Hindu’ has been used as ‘hi’ and ‘ndu’, meaning cow-worshippers. In the same way (as is, in the stated acronym style), this has also been mentioned in the Atharvaveda.'),
 Document(id='f05884a1-91d6-4926-af9f-8a2f9d2c5743', metadata={'source': 'Speech.txt'}, page_content='Taking into view, the region from Himalaya (Mansarovar) to Indu Saro

In [8]:
#Saving to the disk
vectordb = Chroma.from_documents(splits,embeddings, persist_directory = "./chroma")

In [11]:
#load from disk

db2 = Chroma(persist_directory="./chroma", embedding_function=embeddings)
docs = db2.similarity_search("Atharvaveda")
print(docs[0].page_content)

In Sharngdhar Padhatti, the word ‘Hindvah’ has been used for people who follow the vedic path.

In the Brihaspati Agama the kshetra or region of Hindusthana has also been mentioned.

In the Ashvamedhika Parva of Mahabharata, the region of ‘Aryavarta’ has been called Hindusthanaa or Sindhusthanaa.


In [13]:
retreiver = vectordb.as_retriever()

retreiver.invoke("Indu")

[Document(id='f05884a1-91d6-4926-af9f-8a2f9d2c5743', metadata={'source': 'Speech.txt'}, page_content='Taking into view, the region from Himalaya (Mansarovar) to Indu Sarovar (Kanyakumari), ‘hi’ and ‘ndu’ combine as an acronym to create the word ‘Hindu’, and define the geographical region is called Hindusthana or Hindustan'),
 Document(id='7b828916-4ce6-4215-a600-c8003da7eb90', metadata={'source': 'Speech.txt'}, page_content='The one who overcomes deficiencies, poverty, meanness, pettiness, is a Hindu. This is the yogic meaning of the word.\n\nThe one who destroys ‘hinata’ or inferiority, is called Hindu.\n\nThe moon is also called Indu and this also gives the name Hindu.\n\nThe word ‘Hindvah’ has been used in the Kalika Puran.'),
 Document(id='8055bba7-0aa7-4dda-bbf9-eae943d29d94', metadata={'source': 'Speech.txt'}, page_content='The word ‘Hindu’ has been derived from the call of the cow called ‘hinkaar’. The cow’s hinkaar has given the name Hindu as per the Rigveda. The word ‘Hindu’ h